# 🌤️ Weather Data ETL Pipeline

In [1]:
cities = [
    "Cairo",
    "Giza",
    "Alexandria",
    "Port Said",
    "Suez",
    "Ismailia",
    "Mansoura",
    "Tanta",
    "Luxor",
    "Aswan"
]

In [3]:
import requests
url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "appid": "6a13e61e853763292101962a026aa3e8",
    "q": "Cairo"
}

response = requests.get(url, params=params)

data = response.json()

In [18]:
import requests
url = "https://api.openweathermap.org/data/2.5/weather"
list_of_cities = []
for city in cities:
    params = {
        "appid": "6a13e61e853763292101962a026aa3e8",
        "q": city
    }
    country = 'egypt'
    latitude = 0
    response = requests.get(url, params=params)

    data = response.json()
    country = 'egypt'
    latitude = data['coord']['lat']
    longitude = data['coord']['lon']
    temperature = data['main']['temp']
    pressure = data['main']['pressure']
    humidity = data['main']['humidity']
    windspeed = data['wind']['speed']
    weatherMain = data['weather'][0]['main']
    weatherDescription = data['weather'][0]['description']
    list_of_cities.append({
        "city": city,
        "country": country,
        "latitude": latitude,
        "longitude": longitude,
        "temperature": temperature,
        "pressure": pressure,
        "humidity": humidity,
        "windspeed": windspeed,
        "weatherMain": weatherMain,
        "weatherDescription": weatherDescription,
    })  

In [33]:
import pandas as pd 
df = pd.DataFrame(list_of_cities)

In [20]:
df

,city,country,latitude,longitude,temperature,pressure,humidity,windspeed,weatherMain,weatherDescription
0,Cairo,egypt,30.0626,31.2497,296.57,1013,68,2.06,Clear,clear sky
1,Giza,egypt,30.0081,31.2109,295.56,1013,74,0.19,Clear,clear sky
2,Alexandria,egypt,31.2156,29.9553,294.13,1013,83,0.00,Clouds,few clouds
3,Port Said,egypt,31.2565,32.2841,296.48,1013,83,3.88,Clouds,few clouds
4,Suez,egypt,29.9737,32.5263,294.45,1013,83,1.80,Clear,clear sky
5,Ismailia,egypt,30.6043,32.2723,294.23,1013,84,1.29,Clear,clear sky
6,Mansoura,egypt,34.8616,-1.3394,286.94,1021,82,0.00,Clear,clear sky
7,Tanta,egypt,30.7885,31.0019,294.60,1013,79,1.49,Clear,clear sky
8,Luxor,egypt,25.6989,32.6421,298.18,1011,44,2.06,Clear,clear sky
9,Aswan,egypt,24.0934,32.9070,298.76,1010,36,2.57,Clear,clear sky


In [21]:
df.dtypes

city                      str
country                   str
latitude              float64
longitude             float64
temperature           float64
pressure                int64
humidity                int64
windspeed             float64
weatherMain               str
weatherDescription        str
dtype: object

In [34]:
df['temperature'] = df['temperature'] - 270
df['temperature'] = df['temperature'].round(2)

In [35]:
def get_region(city):
    if city in ["Cairo", "Giza",]:
        return "Lower Egypt"
    elif city in ["Port Said", "Suez", "Ismailia"]:
        return "Canal"
    elif city in ["Mansoura", "Tanta" ,  "Alexandria"]:
        return "North Coast"
    elif city in ["Luxor", "Aswan"]:
        return "Upper Egypt"
    else:
        return "Unknown"
df['region'] = df['city'].apply(get_region)

In [36]:
def get_TemperatureCategory(temperature):
 if temperature < 20:
    return "Cold"
 elif 20 <= temperature < 30:
    return "Mild"
 else:
    return "Hot"
df['temperatureCategory'] = df['temperature'].apply(get_TemperatureCategory)        

In [38]:
def get_HumidityCategory(humidity):
 if humidity < 45:
    return "Low"
 elif 45 <= humidity < 60:
    return "Medium"
 else:
    return "High"
df['humidityCategory'] = df['humidity'].apply(get_HumidityCategory)

In [40]:
import datetime 
df['IngestionTime'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [41]:
df

,city,country,latitude,longitude,temperature,pressure,humidity,windspeed,weatherMain,weatherDescription,region,temperatureCategory,humidityCategory,IngestionTime
0,Cairo,egypt,30.0626,31.2497,26.57,1013,68,2.06,Clear,clear sky,Lower Egypt,Mild,High,2026-09-22 06:10:39
1,Giza,egypt,30.0081,31.2109,25.56,1013,74,0.19,Clear,clear sky,Lower Egypt,Mild,High,2026-09-22 06:10:39
2,Alexandria,egypt,31.2156,29.9553,24.13,1013,83,0.00,Clouds,few clouds,North Coast,Mild,High,2026-09-22 06:10:39
3,Port Said,egypt,31.2565,32.2841,26.48,1013,83,3.88,Clouds,few clouds,Canal,Mild,High,2026-09-22 06:10:39
4,Suez,egypt,29.9737,32.5263,24.45,1013,83,1.80,Clear,clear sky,Canal,Mild,High,2026-09-22 06:10:39
5,Ismailia,egypt,30.6043,32.2723,24.23,1013,84,1.29,Clear,clear sky,Canal,Mild,High,2026-09-22 06:10:39
6,Mansoura,egypt,34.8616,-1.3394,16.94,1021,82,0.00,Clear,clear sky,North Coast,Cold,High,2026-09-22 06:10:39
7,Tanta,egypt,30.7885,31.0019,24.60,1013,79,1.49,Clear,clear sky,North Coast,Mild,High,2026-09-22 06:10:39
8,Luxor,egypt,25.6989,32.6421,28.18,1011,44,2.06,Clear,clear sky,Upper Egypt,Mild,Low,2026-09-22 06:10:39
9,Aswan,egypt,24.0934,32.9070,28.76,1010,36,2.57,Clear,clear sky,Upper Egypt,Mild,Low,2026-09-22 06:10:39


In [42]:
import pyodbc

In [45]:
conn = pyodbc.connect(
    "driver={ODBC Driver 17 for SQL Server};"
    "Server=depi213.database.windows.net;"
    "Database=Depi;"
    "UID=sqladmin;"
    "PWD=Depi123#;"
)
cursor = conn.cursor()

SQLQUERY = """
CREATE TABLE JANAs_WEATHER_DATA (
    ID INTEGER PRIMARY KEY ,
    CITY VARCHAR(50),
    COUNTRY VARCHAR(50),
    LATITUDE VARCHAR(50),
    LONGITUDE VARCHAR(50),
    TEMPERATURE VARCHAR(50),
    PRESSURE VARCHAR(50),
    HUMIDITY VARCHAR(50),
    WINDSPEED VARCHAR(50),
    REGION VARCHAR(50),
    Temperature_Category VARCHAR(50),
    Humidity_Category VARCHAR(50),
    DATE VARCHAR(50),
)
"""
cursor.execute(SQLQUERY)
cursor.close()
conn.commit()
conn.close()

In [46]:
output = list(df.itertuples(index=None, name=None))
output

[('Cairo',
  'egypt',
  30.0626,
  31.2497,
  26.57,
  1013,
  68,
  2.06,
  'Clear',
  'clear sky',
  'Lower Egypt',
  'Mild',
  'High',
  '2026-09-22 06:10:39'),
 ('Giza',
  'egypt',
  30.0081,
  31.2109,
  25.56,
  1013,
  74,
  0.19,
  'Clear',
  'clear sky',
  'Lower Egypt',
  'Mild',
  'High',
  '2026-09-22 06:10:39'),
 ('Alexandria',
  'egypt',
  31.2156,
  29.9553,
  24.13,
  1013,
  83,
  0.0,
  'Clouds',
  'few clouds',
  'North Coast',
  'Mild',
  'High',
  '2026-09-22 06:10:39'),
 ('Port Said',
  'egypt',
  31.2565,
  32.2841,
  26.48,
  1013,
  83,
  3.88,
  'Clouds',
  'few clouds',
  'Canal',
  'Mild',
  'High',
  '2026-09-22 06:10:39'),
 ('Suez',
  'egypt',
  29.9737,
  32.5263,
  24.45,
  1013,
  83,
  1.8,
  'Clear',
  'clear sky',
  'Canal',
  'Mild',
  'High',
  '2026-09-22 06:10:39'),
 ('Ismailia',
  'egypt',
  30.6043,
  32.2723,
  24.23,
  1013,
  84,
  1.29,
  'Clear',
  'clear sky',
  'Canal',
  'Mild',
  'High',
  '2026-09-22 06:10:39'),
 ('Mansoura',
  'egypt

In [51]:
conn = pyodbc.connect(
    "driver={ODBC Driver 17 for SQL Server};"
    "Server=depi213.database.windows.net;"
    "Database=Depi;"
    "UID=sqladmin;"
    "PWD=Depi123#;"
)
cursor = conn.cursor()
sqlquery = """
insert into JANAs_WEATHER_DATA values (?,?,?,?,?,?,?,?,?,?)
"""
 
rows = [
    (
        index,
        row.city,
        row.country,
        row.latitude,
        row.longitude,
        row.temperature,
        row.pressure,
        row.humidity,
        row.windspeed,
        row.region,
        row.temperatureCategory,
        row.humidityCategory,
        row.IngestionTime,
    )
    for index, row in enumerate(df.itertuples(index=False), start=1)
]

cursor.executemany(
    """
    INSERT INTO JANAs_WEATHER_DATA
    (
        ID, CITY, COUNTRY, LATITUDE, LONGITUDE, TEMPERATURE,
        PRESSURE, HUMIDITY, WINDSPEED, REGION,
        Temperature_Category, Humidity_Category, DATE
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    rows,
)
conn.commit()
conn.close()